In [3]:
import datetime
import findspark
import pandas as pd
findspark.init()

import sys
print(sys.version)

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

import requests

spark = (
    SparkSession.builder
    .appName("Fuel Prices in Brazil")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

df = spark.read.parquet("databases/fuel_prices/silver/fuels_prices")

df = df.dropna(how='all')

df = df.withColumn('dt_date_month_start', to_date(concat_ws('-', col('dt_year'), col('dt_month'), lit('01'))))

df = (
    df
    .groupBy('nm_region', 'nm_state', 'nm_city', 'nm_fuel_type', 'dt_date_month_start', 'nm_fuel_brand', 'uf_city')
    .agg(
        mean('nu_fuel_price').alias('avg_fuel_price')
    )
    .withColumn("avg_fuel_price_r", round(col("avg_fuel_price"), 2))
    .orderBy('dt_date_month_start', 'nm_state', 'nm_city', 'nm_fuel_type')
)


3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


In [5]:
most_recent_date = df.select('dt_date_month_start').distinct().orderBy(col('dt_date_month_start').desc()).first()['dt_date_month_start'].strftime('%Y-%m-%d')

rows = df.select(
    'dt_date_month_start',
    'avg_fuel_price'
).collect()

list_of_dates = [row.dt_date_month_start for row in rows]
list_of_values = [row.avg_fuel_price for row in rows]

dates_as_strings = [
    d.isoformat() if isinstance(d, (datetime.date, datetime.datetime)) else d 
    for d in list_of_dates
]

payload = {
    'dates': dates_as_strings,
    'values': list_of_values,
    'currency': 'BRL',
    'present_date': most_recent_date
}

headers = {
    "Content-Type": "application/json"
}

values_inflation_adjusted = requests.post('https://financial-utilities-api.onrender.com/inflation_adjustment', json=payload, headers=headers)

if values_inflation_adjusted.status_code != 200:
    values_inflation_adjusted = requests.post('https://financial-utilities-api.onrender.com/inflation_adjustment', json=payload, headers=headers)

df_inflation_adjusted_values = spark.createDataFrame(pd.DataFrame(values_inflation_adjusted.json()))

df_inflation_adjusted_values = df_inflation_adjusted_values.toDF('date', 'original_value', 'inflation_adjusted_avg_fuel_price', 'inflation_perc')

df_inflation_adjusted_values = (
    df_inflation_adjusted_values
    .withColumn("original_value", round(col("original_value"), 2))
)

df = (
    df
    .withColumn("avg_fuel_price", round(col("avg_fuel_price"), 2))
)


Py4JJavaError: An error occurred while calling o188.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 17.0 failed 1 times, most recent failure: Lost task 0.0 in stage 17.0 (TID 23) (JoaoPaiva executor driver): java.io.FileNotFoundException: C:\Users\joaov\AppData\Local\Temp\blockmgr-d76eb9cc-3b11-4324-8bd6-3f7fd6730781\15\temp_shuffle_e49b868c-3d99-42a7-81f0-b510c1083b68 (O sistema não pode encontrar o caminho especificado)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:293)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:235)
	at org.apache.spark.storage.DiskBlockObjectWriter.initialize(DiskBlockObjectWriter.scala:148)
	at org.apache.spark.storage.DiskBlockObjectWriter.open(DiskBlockObjectWriter.scala:168)
	at org.apache.spark.storage.DiskBlockObjectWriter.write(DiskBlockObjectWriter.scala:333)
	at org.apache.spark.shuffle.sort.B

In [ ]:
df.limit(5).show()

Py4JJavaError: An error occurred while calling o168.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 16.0 failed 1 times, most recent failure: Lost task 0.0 in stage 16.0 (TID 22) (192.168.2.3 executor driver): java.io.FileNotFoundException: C:\Users\joaov\AppData\Local\Temp\blockmgr-23ead533-a632-4f98-9092-b52d3132e828\09\temp_shuffle_3b1e28ec-fa04-4412-84d6-aeb792122a9b (O sistema não pode encontrar o caminho especificado)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:293)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:235)
	at org.apache.spark.storage.DiskBlockObjectWriter.initialize(DiskBlockObjectWriter.scala:148)
	at org.apache.spark.storage.DiskBlockObjectWriter.open(DiskBlockObjectWriter.scala:168)
	at org.apache.spark.storage.DiskBlockObjectWriter.write(DiskBlockObjectWriter.scala:333)
	at org.apache.spark.shuffle.sort.Bypa

In [ ]:
rows_inf_adj = df.select(
    'inflation_adjusted_avg_fuel_price'
).collect()

list_inf_adj_values = [row.inflation_adjusted_avg_fuel_price for row in rows_inf_adj]

df.withColumn('inflation_adjusted_avg_fuel_price', list_inf_adj_values).show()

{"ts": "2025-12-31 17:54:21.226", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `inflation_adjusted_avg_fuel_price` cannot be resolved. Did you mean one of the following? [`avg_fuel_price`, `avg_fuel_price_r`, `dt_date_month_start`, `nm_fuel_brand`, `nm_fuel_type`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor8.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o143.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `inflation_adjusted_avg_fuel_price` cannot be resolved. Did you mean one of the following? [`avg_fuel_price`, `avg_fuel_price_r`, `dt_date_month_start`, `nm_fuel_brand`, `nm_fuel_type`]. SQLSTATE: 42703;\n'P

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `inflation_adjusted_avg_fuel_price` cannot be resolved. Did you mean one of the following? [`avg_fuel_price`, `avg_fuel_price_r`, `dt_date_month_start`, `nm_fuel_brand`, `nm_fuel_type`]. SQLSTATE: 42703;
'Project ['inflation_adjusted_avg_fuel_price]
+- Project [nm_region#0, nm_state#1, nm_city#2, nm_fuel_type#5, dt_date_month_start#15, nm_fuel_brand#9, uf_city#13, round(avg_fuel_price#16, 2) AS avg_fuel_price#55, avg_fuel_price_r#33]
   +- Sort [dt_date_month_start#15 ASC NULLS FIRST, nm_state#1 ASC NULLS FIRST, nm_city#2 ASC NULLS FIRST, nm_fuel_type#5 ASC NULLS FIRST], true
      +- Project [nm_region#0, nm_state#1, nm_city#2, nm_fuel_type#5, dt_date_month_start#15, nm_fuel_brand#9, uf_city#13, avg_fuel_price#16, round(avg_fuel_price#16, 2) AS avg_fuel_price_r#33]
         +- Aggregate [nm_region#0, nm_state#1, nm_city#2, nm_fuel_type#5, dt_date_month_start#15, nm_fuel_brand#9, 

In [ ]:
df = (
    df
    .join(
        df_inflation_adjusted_values.select(
            'date',
            'original_value',
            'inflation_adjusted_avg_fuel_price'
        ),
        (df.dt_date_month_start == df_inflation_adjusted_values.date) &
        (df.avg_fuel_price == df_inflation_adjusted_values.original_value),
        how='left'
    )
)

df = df.drop('original_value', 'avg_fuel_price_r', 'date')

Py4JJavaError: An error occurred while calling o165.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 15.0 failed 1 times, most recent failure: Lost task 0.0 in stage 15.0 (TID 19) (192.168.2.3 executor driver): java.io.FileNotFoundException: C:\Users\joaov\AppData\Local\Temp\blockmgr-23ead533-a632-4f98-9092-b52d3132e828\25\temp_shuffle_9bad8e8c-3867-4fe5-a1c4-d112325e41fa (O sistema não pode encontrar o caminho especificado)
	at java.base/java.io.FileOutputStream.open0(Native Method)
	at java.base/java.io.FileOutputStream.open(FileOutputStream.java:293)
	at java.base/java.io.FileOutputStream.<init>(FileOutputStream.java:235)
	at org.apache.spark.storage.DiskBlockObjectWriter.initialize(DiskBlockObjectWriter.scala:148)
	at org.apache.spark.storage.DiskBlockObjectWriter.open(DiskBlockObjectWriter.scala:168)
	at org.apache.spark.storage.DiskBlockObjectWriter.write(DiskBlockObjectWriter.scala:333)
	at org.apache.spark.shuffle.sort.Bypa

In [ ]:
df.write.mode("overwrite").format("parquet").save("databases/fuel_prices/gold/fuels_prices")